#Capa Silver: Limpieza, Calidad de Datos y Tipado

**Objetivo:** Consumir los datos crudos en formato Delta desde la capa Bronze (`ventas_raw` y `tipo_cambio_raw`), aplicar filtrado de calidad de datos (registros sin nulos en campos clave, precios/cantidades válidas y exclusión de monedas inválidas), castear los tipos de datos y aplanar/estructurar los datos de la API de cotizaciones.

**Salida:** `abfss://tpi-data@sttpiicaro.dfs.core.windows.net/silver/` (`ventas_silver` y `tasas_cambio_silver`)

In [0]:
from pyspark.sql import functions as F

# -------------------------------------------------------------------------
# Configuración de Conexión a Azure Blob Storage / Data Lake Gen2
# -------------------------------------------------------------------------
STORAGE_ACCOUNT = "sttpiicaro"
CONTAINER = "tpi-data"

# Clave de acceso a la cuenta de almacenamiento
storage_key = "TU_CLAVE_AQUÍ"

# Credenciales en la sesión de Spark
spark.conf.set(
    f"fs.azure.account.key.{STORAGE_ACCOUNT}.dfs.core.windows.net",
    storage_key
)

# Ruta base ABFSS
BASE = f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net"

# -------------------------------------------------------------------------
# 1. Lectura de Capa Bronze (Formato Delta)
# -------------------------------------------------------------------------

# Lectura de la tabla Delta de ventas desde la carpeta bronze
df_ventas_raw = (
    spark.read
    .format("delta")
    .load(f"{BASE}/bronze/ventas_raw")
)

# Lectura de la tabla / carpeta de tasas de cambio desde bronze
# Si se guardó como Delta:
try:
    df_tasas_raw = spark.read.format("delta").load(f"{BASE}/bronze/tipo_cambio_raw")
except Exception:
    # Alternativa en caso de leer el JSON original directamente
    df_tasas_raw = spark.read.option("multiline", "true").json(f"{BASE}/bronze/tasas_cambio")

# -------------------------------------------------------------------------
# 2. Limpieza y Casteo de Tipos de Datos
# -------------------------------------------------------------------------
df_ventas_clean = (
    df_ventas_raw
    .withColumn("transaction_date", F.to_date(F.col("transaction_date")))
    .withColumn("quantity", F.col("quantity").cast("int"))
    .withColumn("unit_price_local", F.col("unit_price_local").cast("double"))
    .withColumn("total_amount_local", F.col("total_amount_local").cast("double"))
)

# -------------------------------------------------------------------------
# 3. Reglas de Calidad de Datos (Data Quality)
# -------------------------------------------------------------------------
dq_ventas_valida = (
    F.col("transaction_id").isNotNull() &
    F.col("transaction_date").isNotNull() &
    F.col("unit_price_local").isNotNull() & (F.col("unit_price_local") > 0) &
    F.col("quantity").isNotNull() & (F.col("quantity") > 0) &
    (F.col("currency") != "XYZ")
)

df_ventas_silver = df_ventas_clean.filter(dq_ventas_valida)
df_ventas_invalidas = df_ventas_clean.filter(~dq_ventas_valida)

# Reporte de Calidad de Datos
cant_validos = df_ventas_silver.count()
cant_invalidos = df_ventas_invalidas.count()
print(f"[Ventas Silver] Registros válidos: {cant_validos} | Registros excluidos: {cant_invalidos}")
if cant_invalidos > 0:
    df_ventas_invalidas.show(truncate=False)

# -------------------------------------------------------------------------
# 4. Unnest / Aplanado de la API Frankfurter (Tasas de Cambio)
# -------------------------------------------------------------------------
if "rates" in df_tasas_raw.columns:
    df_tasas_plano = (
        df_tasas_raw
        .select(F.explode("rates").alias("fecha_str", "monedas"))
        .select(
            F.to_date(F.col("fecha_str")).alias("fecha"),
            F.explode("monedas").alias("moneda", "tasa_cambio")
        )
    )
else:
    df_tasas_plano = df_tasas_raw

# -------------------------------------------------------------------------
# 5. Escritura a Capa Silver en Formato Delta (tpi-data/silver)
# -------------------------------------------------------------------------
df_ventas_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(f"{BASE}/silver/ventas_silver")

df_tasas_plano.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(f"{BASE}/silver/tasas_cambio_silver")

[Ventas Silver] Registros válidos: 147 | Registros excluidos: 3
+--------------+----------------+-----------+--------------------------+-----------+--------+--------+----------------+------------------+
|transaction_id|transaction_date|customer_id|product_name              |category   |quantity|currency|unit_price_local|total_amount_local|
+--------------+----------------+-----------+--------------------------+-----------+--------+--------+----------------+------------------+
|TXN-00015     |2024-01-30      |CUST-0114  |Noise Canceling Headphones|Electronics|-2      |JPY     |180.38          |-360.76           |
|TXN-00030     |2024-01-21      |CUST-0122  |Wireless Mouse            |Electronics|2       |XYZ     |24.49           |48.98             |
|TXN-00050     |2024-01-09      |CUST-0116  |Monitor 27-inch           |Electronics|1       |CHF     |NULL            |NULL              |
+--------------+----------------+-----------+--------------------------+-----------+--------+--------+